In [1]:
!pip install diffusers transformers accelerate peft bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 32.2 MB/s  0:00:01 eta 0:00:01


## **Updating torchao**

In [2]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 39.2 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


## **The LoRA Fine-Tuning**

In [3]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from diffusers import StableDiffusionPipeline, DDPMScheduler
from diffusers.utils import make_image_grid
from accelerate import Accelerator
from peft import LoraConfig, get_peft_model
from transformers import CLIPTextModel, CLIPTokenizer
from diffusers import AutoencoderKL, UNet2DConditionModel


torch.cuda.empty_cache()

# --- CONFIG ---
MODEL_ID = "runwayml/stable-diffusion-v1-5"
OUTPUT_DIR = "lora_mel_adapter"
REAL_IMG_DIR = "/kaggle/input/datasets/ittisamurtunib/dataset/ISIC_2019_Training_Input/ISIC_2019_Training_Input"
REAL_MEL_CSV = "/kaggle/input/datasets/ittisamurtunib/dataset/CSVs-20260711T054424Z-2-001/CSVs/expA_train.csv"

device = "cuda"
accelerator = Accelerator(mixed_precision="fp16")

# 1. Load Base Components
tokenizer = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(MODEL_ID, subfolder="text_encoder").to(device)
vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae").to(device)
vae.enable_slicing()
vae.enable_tiling()
unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder="unet")
noise_scheduler = DDPMScheduler.from_pretrained(MODEL_ID, subfolder="scheduler")

# Freeze base models
vae.requires_grad_(False)
text_encoder.requires_grad_(False)
unet.requires_grad_(False)

# 2. Apply LoRA to UNet
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,                   # Increase from 4 to 16
    lora_alpha=16,          # Match r
    target_modules=["to_k", "to_q", "to_v", "to_out.0"],
    bias="none",
)

# Ensure unet is in train mode and set up properly before wrapping
unet.enable_gradient_checkpointing()
unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

# 3. Dataset (Real MEL images only)
class MELDataset(Dataset):
    def __init__(self, csv_path, img_dir):
        df = pd.read_csv(csv_path)
        self.images = df[df['label'] == 'MEL']['image'].tolist()
        self.img_dir = img_dir
        self.transform = transforms.Compose([
            transforms.Resize((512, 512)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.img_dir, f"{self.images[idx]}.jpg")).convert("RGB")
        return self.transform(img)

dataset = MELDataset(REAL_MEL_CSV, REAL_IMG_DIR)
dataloader = DataLoader(dataset, batch_size=1, shuffle=True) # Small batch for T4

# 4. Training Loop (Simplified)
optimizer = torch.optim.AdamW(unet.parameters(), lr=1e-5) # Drop from 1e-4 to 1e-5

unet, dataloader, optimizer = accelerator.prepare(unet, dataloader, optimizer)

# Prompt for melanoma
prompt_text = "a dermoscopic image of a melanoma skin lesion, highly detailed, medical photography, realistic"
prompt = tokenizer(prompt_text, 
                   padding="max_length", max_length=tokenizer.model_max_length, 
                   truncation=True, return_tensors="pt").input_ids.to(device)
prompt_embeds = text_encoder(prompt)[0]

num_epochs = 10 # Drop from 50 to 10. (35 images * 10 epochs = 350 steps)
unet.train()

for epoch in range(num_epochs):
    for batch in dataloader:
        with accelerator.accumulate(unet):
            latents = vae.encode(batch.to(device)).latent_dist.sample() * 0.18215
            noise = torch.randn_like(latents)
            timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=device).long()
            
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
            noise_pred = unet(noisy_latents, timesteps, prompt_embeds).sample
            
            loss = torch.nn.functional.mse_loss(noise_pred, noise)
            accelerator.backward(loss)
            optimizer.step()
            optimizer.zero_grad()
            
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1} Loss: {loss.item():.4f}")

# Save LoRA weights
unet = accelerator.unwrap_model(unet)
unet.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to {OUTPUT_DIR}")

Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


tokenizer_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.06M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

... (8 outputs left out)

unet/diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B / 3.44GB            

unet/diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

scheduler_config.json:   0%|          | 0.00/308 [00:00<?, ?B/s]

trainable params: 3,188,736 || all params: 862,709,700 || trainable%: 0.3696
Epoch 10 Loss: 0.0390
LoRA adapter saved to lora_mel_adapter


## **Generation and Filtering**

In [5]:
from peft import PeftModel

# Load pipeline and DISABLE safety checker IMMEDIATELY
pipe = StableDiffusionPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(device)
pipe.safety_checker = None
pipe.requires_safety_checker = False

# Load adapter
pipe.unet = PeftModel.from_pretrained(pipe.unet, OUTPUT_DIR)

os.makedirs("/kaggle/working/synthetic_mel_lora", exist_ok=True)
print("Generating 200 images...")

for i in range(200):
    image = pipe(
        "a dermoscopic image of a melanoma skin lesion, highly detailed, medical photography, realistic",
        negative_prompt="blurry, low quality, cartoon, drawing, text, watermark, black and white",
        num_inference_steps=30
    ).images[0]
    image.save(f"/kaggle/working/synthetic_mel_lora/lora_mel_{i:04d}.jpg")

manifest = pd.DataFrame({
    'image': [f"lora_mel_{i:04d}" for i in range(200)],
    'label': ['MEL'] * 200,
    'source': ['lora_synthetic'] * 200
})
manifest.to_csv("/kaggle/working/lora_manifest.csv", index=False)
print("Generation complete!")

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

... (199 outputs left out)

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

Generation complete!


## **Feature-Space Filtering LoRA generation**

In [11]:
# filter_synthetic.py

import os
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torchvision import models, transforms


# --- CONFIG ---
SSL_CKPT = '/kaggle/input/datasets/ittisamurtunib/ssl-encoder-real-24000/ssl_encoder_real_24000.pth'  # Stage-1 checkpoint
REAL_MEL_CSV = '/kaggle/input/datasets/ittisamurtunib/dataset/CSVs-20260711T054424Z-2-001/CSVs/expA_train.csv'          # the split file with real MEL images
REAL_IMG_DIR = '/kaggle/input/datasets/ittisamurtunib/dataset/ISIC_2019_Training_Input/ISIC_2019_Training_Input'     # folder with real images
SYNTH_DIR = '/kaggle/working/synthetic_mel_lora'       # Synthetic LoRA Image Dir
SYNTH_MANIFEST = '/kaggle/working/lora_manifest.csv' 

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- DEFINE ARCHITECTURES TO MATCH CHECKPOINT ---
class SimpleEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 2
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Block 3
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            # Pooling
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )
        self.fc = nn.Linear(128, out_dim)

    def forward(self, x):
        h = self.features(x)
        z = self.fc(h)
        return z

class SimCLR(nn.Module):
    def __init__(self, encoder, projection_dim=64):
        super().__init__()
        self.encoder = encoder
        self.projector = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, projection_dim)
        )
    def forward(self, x):
        h = self.encoder(x)
        z = F.normalize(self.projector(h), dim=1)
        return h, z

# --- LOAD ENCODER ROBUSTLY ---
encoder = SimpleEncoder()
state = torch.load(SSL_CKPT, map_location=DEVICE)

# Clean up prefixes if saved from a wrapped model
new_state = {}
for k, v in state.items():
    k = k.replace('encoder.', '').replace('backbone.', '')
    new_state[k] = v

encoder.load_state_dict(new_state, strict=False)
encoder.to(DEVICE).eval()

# --- TRANSFORM (same as fine-tuning) ---
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def extract_features(img_paths):
    feats = []
    with torch.no_grad():
        for path in img_paths:
            img = Image.open(path).convert('RGB')
            tensor = transform(img).unsqueeze(0).to(DEVICE)
            feat = encoder(tensor).cpu()
            feats.append(F.normalize(feat, dim=1))
    return torch.cat(feats, dim=0)

# --- LOAD REAL MEL IMAGES ---
df_train = pd.read_csv(REAL_MEL_CSV)
real_mel_df = df_train[df_train['label'] == 'MEL']
real_paths = [os.path.join(REAL_IMG_DIR, f"{img}.jpg") for img in real_mel_df['image']]
real_feats = extract_features(real_paths)  # shape (35, 128)

# --- LOAD SYNTHETIC IMAGES ---
synth_df = pd.read_csv(SYNTH_MANIFEST)
synth_paths = [os.path.join(SYNTH_DIR, f"{row['image']}.jpg") for _, row in synth_df.iterrows()]
synth_feats = extract_features(synth_paths)  # shape (200, 128)

# --- COMPUTE NEAREST-NEIGHBOUR SIMILARITY ---
# cosine similarity: real_feats (35,128) x synth_feats.T (128,200)
sim_matrix = real_feats @ synth_feats.T  # (35, 200)
max_sim_per_synth, _ = sim_matrix.max(dim=0)  # best match for each synth

# --- FILTER ---
THRESHOLD = 0.90
keep_mask = (max_sim_per_synth < THRESHOLD).cpu().numpy()
kept_indices = synth_df.index[keep_mask].tolist()
rejected_indices = synth_df.index[~keep_mask].tolist()

print(f"Total synthetic: {len(synth_df)}")
print(f"Kept (diverse): {len(kept_indices)}")
print(f"Rejected (near-duplicates): {len(rejected_indices)}")

# Save the filtered list for the next training run
synth_df_kept = synth_df.iloc[kept_indices]

import os
# Ensure the output directory exists
os.makedirs("outputs", exist_ok=True)

# Save the filtered list
synth_df_kept.to_csv('outputs/synthetic_mel_filtered.csv', index=False)

Total synthetic: 200
Kept (diverse): 83
Rejected (near-duplicates): 117


## **Training Classifier** 

In [15]:
import gc
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import os
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import f1_score, average_precision_score, recall_score, precision_score, accuracy_score

# --- SAFELY FREE VRAM ---
try:
    del pipe, unet, vae, text_encoder
except NameError:
    pass # Variables don't exist, which is fine
gc.collect()
torch.cuda.empty_cache()

# --- CONFIG ---
CLASSES = ['BKL', 'MEL', 'NV']
SEEDS = [42, 123, 2024]
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BACKUP_DIR = '/kaggle/input/datasets/ittisamurtunib/dataset/CSVs-20260711T054424Z-2-001/CSVs'
REAL_IMG_DIR = '/kaggle/input/datasets/ittisamurtunib/dataset/ISIC_2019_Training_Input/ISIC_2019_Training_Input'
LORA_IMG_DIR = '/kaggle/working/synthetic_mel_lora/'
SSL_CKPT = '/kaggle/input/datasets/ittisamurtunib/ssl-encoder-real-24000/ssl_encoder_real_24000.pth'

# --- MODEL DEFINITIONS ---
class SimpleEncoder(nn.Module):
    def __init__(self, out_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten()
        )
        self.fc = nn.Linear(128, out_dim)
    def forward(self, x):
        return self.fc(self.features(x))

class SSLClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3), nn.Linear(64, num_classes))
    def forward(self, x):
        return self.classifier(self.encoder(x))

class FocalLoss(nn.Module):
    def __init__(self, alpha, gamma=1.5):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, targets):
        logp = F.log_softmax(logits, dim=1)
        p = logp.exp()
        logp_t = logp.gather(1, targets.unsqueeze(1)).squeeze(1)
        p_t = p.gather(1, targets.unsqueeze(1)).squeeze(1)
        return (-self.alpha[targets] * (1 - p_t) ** self.gamma * logp_t).mean()

class CombinedDataset(Dataset):
    def __init__(self, df, training=True):
        self.df = df.reset_index(drop=True)
        self.training = training
        self.class_to_idx = {c: i for i, c in enumerate(CLASSES)}
        self.transform_std = transforms.Compose([
            transforms.RandomHorizontalFlip(), transforms.RandomRotation(15),
            transforms.Resize((224, 224)), transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.transform_mel = transforms.Compose([
            transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
            transforms.RandomRotation(45), transforms.ColorJitter(0.4, 0.4, 0.3, 0.1),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.85, 1.15)),
            transforms.Resize((224, 224)), transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        self.transform_eval = transforms.Compose([
            transforms.Resize((224, 224)), transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row['image']
        # Check if path is absolute (synthetic) or relative (real)
        if not os.path.isabs(img_path):
            img_path = os.path.join(REAL_IMG_DIR, f"{img_path}.jpg")
        
        img = Image.open(img_path).convert('RGB')
        label = self.class_to_idx[row['label']]
        tfm = (self.transform_mel if row['label'] == 'MEL' else self.transform_std) if self.training else self.transform_eval
        return tfm(img), label

# --- PREPARE DATA ---
train_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_train.csv'))
val_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_val.csv'))
test_df = pd.read_csv(os.path.join(BACKUP_DIR, 'expA_test.csv'))

# Load filtered LoRA images
lora_df = pd.read_csv('/kaggle/working/outputs/synthetic_mel_filtered.csv')
lora_df['image'] = lora_df['image'].apply(lambda x: os.path.join(LORA_IMG_DIR, f"{x}.jpg"))
lora_df['label'] = 'MEL'

# Combine Real + Synthetic
combined_train_df = pd.concat([train_df, lora_df], ignore_index=True)
print(f"Training on {len(combined_train_df)} images ({len(train_df)} real, {len(lora_df)} synthetic)")

# --- TRAINING LOOP ---
results = []
for seed in SEEDS:
    torch.manual_seed(seed)
    np.random.seed(seed)
    
    train_loader = DataLoader(CombinedDataset(combined_train_df, training=True), batch_size=16, shuffle=True)
    val_loader = DataLoader(CombinedDataset(val_df, training=False), batch_size=16, shuffle=False)
    test_loader = DataLoader(CombinedDataset(test_df, training=False), batch_size=16, shuffle=False)
    
    encoder = SimpleEncoder()
    state = torch.load(SSL_CKPT, map_location=DEVICE)
    state = {k.replace('encoder.', ''): v for k, v in state.items()}
    encoder.load_state_dict(state, strict=False)
    
    model = SSLClassifier(encoder, 3).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    focal = FocalLoss(torch.tensor([2.0, 4.0, 1.0], device=DEVICE))
    
    best_f1, best_state, no_improve = -1, None, 0
    for epoch in range(30):
        model.train()
        for images, targets in train_loader:
            optimizer.zero_grad()
            loss = focal(model(images.to(DEVICE)), targets.to(DEVICE))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
        model.eval()
        val_preds, val_targets = [], []
        with torch.no_grad():
            for images, targets in val_loader:
                val_preds.extend(model(images.to(DEVICE)).argmax(dim=1).cpu().numpy())
                val_targets.extend(targets.numpy())
        
        val_f1 = f1_score(val_targets, val_preds, average='macro', zero_division=0)
        if val_f1 > best_f1:
            best_f1, best_state, no_improve = val_f1, copy.deepcopy(model.state_dict()), 0
        else:
            no_improve += 1
            if no_improve >= 7:
                break
                
    model.load_state_dict(best_state)
    model.eval()
    test_probs, test_targets = [], []
    with torch.no_grad():
        for images, targets in test_loader:
            test_probs.append(F.softmax(model(images.to(DEVICE)), dim=1).cpu().numpy())
            test_targets.extend(targets.numpy())
            
    test_probs = np.concatenate(test_probs)
    test_targets = np.array(test_targets)
    test_preds = test_probs.argmax(axis=1)
    
    mel_true = (test_targets == 1).astype(int)
    mel_pred = (test_preds == 1).astype(int)
    
    res = {
        'seed': seed,
        'acc': accuracy_score(test_targets, test_preds),
        'f1': f1_score(test_targets, test_preds, average='macro', zero_division=0),
        'mel_rec': recall_score(mel_true, mel_pred, zero_division=0),
        'mel_prec': precision_score(mel_true, mel_pred, zero_division=0),
        'mel_ap': average_precision_score(mel_true, test_probs[:, 1])
    }
    print(f"Seed {seed}: Acc={res['acc']:.3f} F1={res['f1']:.3f} MEL_Rec={res['mel_rec']:.3f} MEL_Prec={res['mel_prec']:.3f} MEL_AP={res['mel_ap']:.3f}")
    results.append(res)

df_results = pd.DataFrame(results)
print("\n--- FINAL RESULTS (LoRA Diffusion + Filter) ---")
print("Mean:")
print(df_results.mean(numeric_only=True).round(3))
print("\nStd Dev:")
print(df_results.std(numeric_only=True).round(3))

Training on 566 images (483 real, 83 synthetic)
Seed 42: Acc=0.750 F1=0.597 MEL_Rec=0.375 MEL_Prec=0.429 MEL_AP=0.238
Seed 123: Acc=0.731 F1=0.482 MEL_Rec=0.375 MEL_Prec=0.375 MEL_AP=0.180
Seed 2024: Acc=0.673 F1=0.481 MEL_Rec=0.250 MEL_Prec=0.286 MEL_AP=0.203

--- FINAL RESULTS (LoRA Diffusion + Filter) ---
Mean:
seed        729.667
acc           0.718
f1            0.520
mel_rec       0.333
mel_prec      0.363
mel_ap        0.207
dtype: float64

Std Dev:
seed        1121.657
acc            0.040
f1             0.067
mel_rec        0.072
mel_prec       0.072
mel_ap         0.029
dtype: float64
